### **Walmart Sales Forecasting - Feature Engineering**

### **1. Running Feature Pipeline**

In [1]:
import pandas as pd
from pathlib import Path

from src.features.build_features import build_features

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

cleaned_data = pd.read_csv(DATA_PROCESSED / "walmart_clean.csv")

cleaned_data["Date"] = pd.to_datetime(cleaned_data["Date"])

In [2]:
feature_data = build_features(cleaned_data) # Build features using the cleaned data

feature_data.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,is_train,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,Month,Week,Day,Total_MarkDown,Holiday_Markdown,Lag_1,Lag_2,Lag_4,Rolling_Mean_4,Rolling_Std_4
0,1,1,2010-02-05,24924.50,False,1,42.31,2.572,0.0,0.0,...,2,5,5,0.0,0.0,0.00,0.00,0.0,0.00,0.000000
1,1,1,2010-02-12,46039.49,True,1,38.51,2.548,0.0,0.0,...,2,6,12,0.0,0.0,24924.50,0.00,0.0,0.00,0.000000
2,1,1,2010-02-19,41595.55,False,1,39.93,2.514,0.0,0.0,...,2,7,19,0.0,0.0,46039.49,24924.50,0.0,0.00,0.000000
3,1,1,2010-02-26,19403.54,False,1,46.63,2.561,0.0,0.0,...,2,8,26,0.0,0.0,41595.55,46039.49,0.0,0.00,0.000000
4,1,1,2010-03-05,21827.90,False,1,46.50,2.625,0.0,0.0,...,3,9,5,0.0,0.0,19403.54,41595.55,24924.5,32990.77,12832.106391


In [3]:
feature_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 536634 entries, 0 to 536633
Data columns (total 28 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Store             536634 non-null  int64         
 1   Dept              536634 non-null  int64         
 2   Date              536634 non-null  datetime64[us]
 3   Weekly_Sales      421570 non-null  float64       
 4   IsHoliday         536634 non-null  bool          
 5   is_train          536634 non-null  int64         
 6   Temperature       536634 non-null  float64       
 7   Fuel_Price        536634 non-null  float64       
 8   MarkDown1         536634 non-null  float64       
 9   MarkDown2         536634 non-null  float64       
 10  MarkDown3         536634 non-null  float64       
 11  MarkDown4         536634 non-null  float64       
 12  MarkDown5         536634 non-null  float64       
 13  CPI               536634 non-null  float64       
 14  Unemployment   

In [4]:
feature_data.isnull().sum().sort_values(ascending=False).head(15) # Check for missing values in the feature data

Weekly_Sales    115064
Store                0
Dept                 0
Date                 0
IsHoliday            0
is_train             0
Temperature          0
Fuel_Price           0
MarkDown1            0
MarkDown2            0
MarkDown3            0
MarkDown4            0
MarkDown5            0
CPI                  0
Unemployment         0
dtype: int64

### **2. Separate Train & Test**

In [5]:
train_data = feature_data[feature_data["is_train"] == 1].copy()
test_data = feature_data[feature_data["is_train"] == 0].copy()

print(train_data.shape)
print(test_data.shape)

(421570, 28)
(115064, 28)


In [6]:
# Check for missing values in the target variable (Weekly_Sales) in both train and test datasets
print("Missing values in train data (Weekly_Sales):", train_data["Weekly_Sales"].isnull().sum())
print("Missing values in test data (Weekly_Sales):", test_data["Weekly_Sales"].isnull().sum())

Missing values in train data (Weekly_Sales): 0
Missing values in test data (Weekly_Sales): 115064


In [7]:
feature_data.to_csv(DATA_PROCESSED / "walmart_features.csv", index=False)

**Feature Engineering Validation**

- The missing values observed in lag and rolling features are expected because the earliest records for each Store-Department combination do not have sufficient historical data.
- The missing values in `Weekly_Sales` correspond to test rows (`is_train = 0`), which are used for final forecasting and therefore do not contain target values.
- These missing values were filled with 0 to represent the absence of prior sales history.